# 03. Model Training, Tuning, and Stratified Cross-Validation

## Overview
In this notebook, we train and tune machine learning models exclusively on the IBM Telco Customer Churn training split (`X_train.csv`, `y_train.csv`).

### Methodological Best Practices:
1. **Leakage-Safe Pipelines**: Preprocessing (scaling, one-hot encoding) and resampling (SMOTE) are embedded inside `imblearn.pipeline.Pipeline` objects to ensure no validation fold information leaks into training.
2. **Stratified 5-Fold Cross-Validation**: Cross-validation folds preserve the target churn class ratio across all splits.
3. **Model Diversity**: We compare:
   - **Logistic Regression** (Interpretable linear baseline)
   - **Random Forest** (Non-linear ensemble of decision trees)
   - **XGBoost Classifier** (Gradient boosted decision trees)
4. **Imbalance Mitigation Comparison**: We systematically compare three strategies per model algorithm:
   - Unweighted baseline (`none`)
   - Algorithmic class weighting (`class_weight='balanced'` / `scale_pos_weight`)
   - Synthetic Over-sampling (`SMOTE` inside training folds)


### Environment Setup and Library Imports
We import scikit-learn model selection utilities, machine learning classifiers, and `imblearn.pipeline.Pipeline` for leak-free resampling.

In [1]:
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Set project root path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
CV_SPLITS = 5

### Load Training Data and Preprocessor Artifact
We load `X_train.csv`, `y_train.csv`, and the preprocessor `ColumnTransformer` fitted in Notebook 01.

In [2]:
X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze("columns").astype(int)
preprocessor = joblib.load(MODEL_DIR / "preprocessor.joblib")

print(f"X_train Shape: {X_train.shape[0]} rows, {X_train.shape[1]} columns")
print(f"Training Target Balance: {y_train.value_counts().to_dict()} (Churn Rate: {y_train.mean():.2%})")

X_train Shape: 5634 rows, 27 columns
Training Target Balance: {0: 4139, 1: 1495} (Churn Rate: 26.54%)


### Configure Stratified 5-Fold Cross-Validation
Stratified K-Fold cross-validation ensures each fold maintains the original ~26.5% churn distribution.

In [3]:
cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)
print(f"Stratified {CV_SPLITS}-Fold CV initialized with random_state={RANDOM_STATE}.")

Stratified 5-Fold CV initialized with random_state=42.


### Leak-Safe Pipeline Construction Helper
We build a generic function `build_model_pipeline(estimator, strategy)` that combines:
1. `preprocessor`: `ColumnTransformer`
2. `smote` (optional): `SMOTE(random_state=42)` included ONLY when strategy == `'smote'`
3. `estimator`: Machine learning algorithm


In [4]:
def build_model_pipeline(estimator, strategy='none'):
    steps = [('preprocess', preprocessor)]
    
    if strategy == 'smote':
        steps.append(('smote', SMOTE(random_state=RANDOM_STATE)))
        
    steps.append(('model', estimator))
    return ImbPipeline(steps)

print("Pipeline helper function established.")

Pipeline helper function established.


### Train and Tune Logistic Regression (Baseline Model)
We evaluate Logistic Regression across `none`, `class_weight='balanced'`, and `smote` strategies.

Adding a warning filter to keep the notebook clean 

In [5]:
import warnings

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=r".*BaseEstimator\._validate_data.*"
)

In [6]:
cv_results_list = []
fitted_pipelines = {}

lr_strategies = {
    'none': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'class_weight': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    'smote': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
}

lr_param_grid = {
    'model__C': [0.01, 0.1, 1.0, 10.0]
}

for strategy_name, estimator in lr_strategies.items():
    pipe = build_model_pipeline(estimator, strategy=strategy_name if strategy_name == 'smote' else 'none')
    
    grid = GridSearchCV(
        pipe, 
        param_grid=lr_param_grid, 
        cv=cv, 
        scoring='roc_auc', 
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    
    # Evaluate best estimator across folds for ROC-AUC and PR-AUC
    scores = cross_validate(
        grid.best_estimator_, 
        X_train, y_train, 
        cv=cv, 
        scoring={'roc_auc': 'roc_auc', 'pr_auc': 'average_precision'},
        n_jobs=-1
    )
    
    candidate_name = f"LogisticRegression__{strategy_name}"
    fitted_pipelines[candidate_name] = grid.best_estimator_
    
    cv_results_list.append({
        'candidate': candidate_name,
        'model': 'LogisticRegression',
        'strategy': strategy_name,
        'best_params': str(grid.best_params_),
        'cv_roc_auc_mean': scores['test_roc_auc'].mean(),
        'cv_roc_auc_std': scores['test_roc_auc'].std(),
        'cv_pr_auc_mean': scores['test_pr_auc'].mean(),
        'cv_pr_auc_std': scores['test_pr_auc'].std()
    })
    print(f"Evaluated {candidate_name} | Best ROC-AUC: {scores['test_roc_auc'].mean():.4f}")

Evaluated LogisticRegression__none | Best ROC-AUC: 0.8609
Evaluated LogisticRegression__class_weight | Best ROC-AUC: 0.8608
Evaluated LogisticRegression__smote | Best ROC-AUC: 0.8596


Evaluated LogisticRegression__class_weight | Best ROC-AUC: 0.8608


Evaluated LogisticRegression__smote | Best ROC-AUC: 0.8596


### Train and Tune Random Forest Classifier
We perform hyperparameter optimization for Random Forest across `none`, `class_weight='balanced'`, and `smote` strategies.

In [7]:
rf_strategies = {
    'none': RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    'class_weight': RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    'smote': RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
}

rf_param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [6, 10, 15, None],
    'model__min_samples_split': [2, 5, 10]
}

for strategy_name, estimator in rf_strategies.items():
    pipe = build_model_pipeline(estimator, strategy=strategy_name if strategy_name == 'smote' else 'none')
    
    search = RandomizedSearchCV(
        pipe, 
        param_distributions=rf_param_grid, 
        n_iter=6, 
        cv=cv, 
        scoring='roc_auc', 
        random_state=RANDOM_STATE, 
        n_jobs=-1
    )
    search.fit(X_train, y_train)
    
    scores = cross_validate(
        search.best_estimator_, 
        X_train, y_train, 
        cv=cv, 
        scoring={'roc_auc': 'roc_auc', 'pr_auc': 'average_precision'},
        n_jobs=-1
    )
    
    candidate_name = f"RandomForest__{strategy_name}"
    fitted_pipelines[candidate_name] = search.best_estimator_
    
    cv_results_list.append({
        'candidate': candidate_name,
        'model': 'RandomForest',
        'strategy': strategy_name,
        'best_params': str(search.best_params_),
        'cv_roc_auc_mean': scores['test_roc_auc'].mean(),
        'cv_roc_auc_std': scores['test_roc_auc'].std(),
        'cv_pr_auc_mean': scores['test_pr_auc'].mean(),
        'cv_pr_auc_std': scores['test_pr_auc'].std()
    })
    print(f"Evaluated {candidate_name} | Best ROC-AUC: {scores['test_roc_auc'].mean():.4f}")


Evaluated RandomForest__none | Best ROC-AUC: 0.8579
Evaluated RandomForest__class_weight | Best ROC-AUC: 0.8594
Evaluated RandomForest__smote | Best ROC-AUC: 0.8576


Evaluated RandomForest__class_weight | Best ROC-AUC: 0.8594


Evaluated RandomForest__smote | Best ROC-AUC: 0.8576


### Train and Tune XGBoost Classifier
We tune gradient-boosted decision trees (`XGBClassifier`) using `scale_pos_weight` and `SMOTE` imbalance strategies.

In [8]:
# XGBoost Model Comparison
# Strategies: None, Class Weight, SMOTE

scale_pos_weight_value = (y_train == 0).sum() / (y_train == 1).sum()

# XGBoost strategies

xgb_strategies = {
    'none': XGBClassifier(
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        device='cpu',
        tree_method='hist',
        n_jobs=1
    ),

    'class_weight': XGBClassifier(
        eval_metric='logloss',
        scale_pos_weight=scale_pos_weight_value,
        random_state=RANDOM_STATE,
        device='cpu',
        tree_method='hist',
        n_jobs=1
    ),

    'smote': XGBClassifier(
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        device='cpu',
        tree_method='hist',
        n_jobs=1
    )
}


# Hyperparameter search space


xgb_param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.03, 0.1],
    'model__subsample': [0.8, 1.0]
}


# Train and evaluate each strategy


for strategy_name, estimator in xgb_strategies.items():

  
    print(f"Training XGBoost: {strategy_name}")
   

    # Build pipeline
    pipe = build_model_pipeline(
        estimator,
        strategy=strategy_name if strategy_name == 'smote' else 'none'
    )

 
# Randomized hyperparameter search
   

    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=xgb_param_grid,
        n_iter=6,
        cv=cv,
        scoring='roc_auc',
        random_state=RANDOM_STATE,
        n_jobs=1,
        error_score='raise'
    )

    # Fit
    search.fit(X_train, y_train)


# Cross-validation evaluation of best model
   
    scores = cross_validate(
        search.best_estimator_,
        X_train,
        y_train,
        cv=cv,
        scoring={
            'roc_auc': 'roc_auc',
            'pr_auc': 'average_precision'
        },
        n_jobs=1
    )

    candidate_name = f"XGBoost__{strategy_name}"

    fitted_pipelines[candidate_name] = search.best_estimator_

   
    cv_results_list.append({
        'candidate': candidate_name,
        'model': 'XGBoost',
        'strategy': strategy_name,
        'best_params': str(search.best_params_),

        'cv_roc_auc_mean':
            scores['test_roc_auc'].mean(),

        'cv_roc_auc_std':
            scores['test_roc_auc'].std(),

        'cv_pr_auc_mean':
            scores['test_pr_auc'].mean(),

        'cv_pr_auc_std':
            scores['test_pr_auc'].std()
    })


    print(f"Best parameters: {search.best_params_}")

    print(
        f"CV ROC-AUC: "
        f"{scores['test_roc_auc'].mean():.4f} "
        f"(± {scores['test_roc_auc'].std():.4f})"
    )

    print(
        f"CV PR-AUC: "
        f"{scores['test_pr_auc'].mean():.4f} "
        f"(± {scores['test_pr_auc'].std():.4f})"
    )

    print(f"Evaluated {candidate_name}")

Training XGBoost: none
Best parameters: {'model__subsample': 1.0, 'model__n_estimators': 100, 'model__max_depth': 7, 'model__learning_rate': 0.03}
CV ROC-AUC: 0.8579 (± 0.0064)
CV PR-AUC: 0.6787 (± 0.0140)
Evaluated XGBoost__none
Training XGBoost: class_weight
Best parameters: {'model__subsample': 1.0, 'model__n_estimators': 100, 'model__max_depth': 7, 'model__learning_rate': 0.03}
CV ROC-AUC: 0.8563 (± 0.0052)
CV PR-AUC: 0.6766 (± 0.0161)
Evaluated XGBoost__class_weight
Training XGBoost: smote
Best parameters: {'model__subsample': 1.0, 'model__n_estimators': 100, 'model__max_depth': 7, 'model__learning_rate': 0.03}
CV ROC-AUC: 0.8550 (± 0.0092)
CV PR-AUC: 0.6677 (± 0.0213)
Evaluated XGBoost__smote


Best parameters: {'model__subsample': 1.0, 'model__n_estimators': 100, 'model__max_depth': 7, 'model__learning_rate': 0.03}
CV ROC-AUC: 0.8579 (± 0.0064)
CV PR-AUC: 0.6787 (± 0.0140)
Evaluated XGBoost__none
Training XGBoost: class_weight


Best parameters: {'model__subsample': 1.0, 'model__n_estimators': 100, 'model__max_depth': 7, 'model__learning_rate': 0.03}
CV ROC-AUC: 0.8563 (± 0.0052)
CV PR-AUC: 0.6766 (± 0.0161)
Evaluated XGBoost__class_weight
Training XGBoost: smote


Best parameters: {'model__subsample': 1.0, 'model__n_estimators': 100, 'model__max_depth': 7, 'model__learning_rate': 0.03}
CV ROC-AUC: 0.8550 (± 0.0092)
CV PR-AUC: 0.6677 (± 0.0213)
Evaluated XGBoost__smote


### Summarize CV Results and Save Best Model Pipelines
We compile the cross-validation comparison table, select the top-performing candidate based on mean CV ROC-AUC, and save the fitted models and metadata for evaluation in Notebook 04.

In [9]:
cv_df = pd.DataFrame(cv_results_list).sort_values('cv_roc_auc_mean', ascending=False).reset_index(drop=True)

print("Cross-Validation Comparison Table")
display(cv_df[['candidate', 'model', 'strategy', 'cv_roc_auc_mean', 'cv_roc_auc_std', 'cv_pr_auc_mean', 'cv_pr_auc_std']])

# Save CV results to CSV
cv_df.to_csv(TABLES_DIR / "cv_comparison.csv", index=False)
cv_df.to_csv(TABLES_DIR / "cv_results.csv", index=False)

# Identify best candidate
best_candidate_row = cv_df.iloc[0]
best_candidate_name = best_candidate_row['candidate']
best_pipeline = fitted_pipelines[best_candidate_name]

print(f"\nSelected Best Candidate Pipeline: '{best_candidate_name}' (CV ROC-AUC: {best_candidate_row['cv_roc_auc_mean']:.4f})")

# Save all fitted pipelines and best model artifact
joblib.dump(fitted_pipelines, MODEL_DIR / "models.joblib")
joblib.dump(best_pipeline, MODEL_DIR / "final_pipeline.joblib")

best_metadata = {
    'candidate_name': best_candidate_name,
    'model_type': best_candidate_row['model'],
    'strategy': best_candidate_row['strategy'],
    'best_params': best_candidate_row['best_params'],
    'cv_roc_auc_mean': float(best_candidate_row['cv_roc_auc_mean']),
    'cv_pr_auc_mean': float(best_candidate_row['cv_pr_auc_mean'])
}

with open(MODEL_DIR / "best_candidate_metadata.json", "w") as f:
    json.dump(best_metadata, f, indent=2)

print("Saved fitted pipelines to models/models.joblib and best candidate metadata to models/best_candidate_metadata.json")


Cross-Validation Comparison Table


,candidate,model,strategy,cv_roc_auc_mean,cv_roc_auc_std,cv_pr_auc_mean,cv_pr_auc_std
0,LogisticRegression__none,LogisticRegression,none,0.860854,0.011855,0.689223,0.015633
1,LogisticRegression__class_weight,LogisticRegression,class_weight,0.860823,0.011818,0.688028,0.016027
2,LogisticRegression__smote,LogisticRegression,smote,0.859617,0.011515,0.686925,0.017653
3,RandomForest__class_weight,RandomForest,class_weight,0.859378,0.008947,0.680755,0.020482
4,RandomForest__none,RandomForest,none,0.857901,0.009770,0.682249,0.021634
5,XGBoost__none,XGBoost,none,0.857876,0.006370,0.678687,0.014003
6,RandomForest__smote,RandomForest,smote,0.857559,0.008935,0.671840,0.020302
7,XGBoost__class_weight,XGBoost,class_weight,0.856302,0.005235,0.676593,0.016141
8,XGBoost__smote,XGBoost,smote,0.855031,0.009188,0.667683,0.021296



Selected Best Candidate Pipeline: 'LogisticRegression__none' (CV ROC-AUC: 0.8609)
Saved fitted pipelines to models/models.joblib and best candidate metadata to models/best_candidate_metadata.json


Saved fitted pipelines to models/models.joblib and best candidate metadata to models/best_candidate_metadata.json
